## __Aprendizaje no supervisado__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

__Asunto__: Robust Covariance

***

In [ ]:
## Librerias
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from pandas import DataFrame
from numpy import array, ones, concatenate, linspace, sqrt
from numpy.random import seed, randn, uniform

from sklearn.model_selection import train_test_split
import matplotlib.transforms as transforms
from matplotlib.patches import Ellipse

from sklearn.covariance import EllipticEnvelope

__Dataset:__

In [ ]:
seed(0)

## Numero de muestras, y de outliers
n_samples, n_outliers = 360, 100

## matriz de covarianza
covariance = array([[0.5, -0.1], [0.7, 0.4]])

## Generación de grupo de puntos
cluster_1 = 0.4 * randn(n_samples, 2) @ covariance + array([2, 2])  # general
cluster_2 = 0.3 * randn(n_samples, 2) + array([-2, -2])  # spherical
outliers = uniform(low=-4, high=4, size=(n_outliers, 2))

## Armado de dataset
X = concatenate([cluster_1, cluster_2, outliers])
y = concatenate(
    [ones((2 * n_samples), dtype=int), -ones((n_outliers), dtype=int)]
)

print('(shape) X: {} - y: {}'.format(X.shape, y.shape))

In [ ]:
## Partición de datos
X_train_Val, X_test, y_train_val, y_test = train_test_split(X, y, 
                                                            test_size=0.121, 
                                                            stratify=y, 
                                                            random_state=9001)

X_train, X_val, y_train, y_val = train_test_split(X_train_Val, y_train_val, 
                                                  test_size=0.138, 
                                                  stratify=y_train_val, 
                                                  random_state=9001)

print('(shape - Train) X: {} - y: {}'.format(X_train.shape, y_train.shape))
print('(shape - Validate) X: {} - y: {}'.format(X_val.shape, y_val.shape))
print('(shape - Test) X: {} - y: {}'.format(X_test.shape, y_test.shape))

__Visualización de los datos considerados__

In [ ]:
plt.figure(figsize=(7, 7))
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                )
plt.title("Nube de puntos")
plt.xlabel('x'), plt.ylabel('y')
plt.tight_layout()
plt.show()

## Clase EllipticEnvelope

```{python}
    EllipticEnvelope(store_precision=True, 
                     assume_centered=False, 
                     support_fraction=None, 
                     contamination=0.1, 
                     random_state=None)
```

| Parámetros | Descripción |
|------------|-------------|
| store_precision | especificar si se almacena la precision estimada (por defecto, True). |
| assume_centered | Si es True, se aplica centralización para luego computar la matriz de covarianza mediante Miminum Covariance Determinat (MCD). Si es False, no se aplica un pre-tratamiento (por defecto, False). |
| support_fraction | es el porcentaje de los puntos para ser incluida en el soporte del computo de MCD. Si es None estima el soporte mínimo como `[n_sample + n_features + 1] / 2` (por defecto, None). |
| contamination | porcentaje de outliers en la data (0, 0.5] (por defecto, 0.1) | 
| random_state | (Optional): semilla de aleatoriedad  |

<br>

| Atributos | Descripción |
|------------|-------------|
| location_ | media estimada. |
| covariance_ | matriz de covarianza robusta estimada. |
| precision_ | retorna la matriz pseudo inverssa estimada (solo se habilita si sotre_precision=True). | 
| support_ | retorna una mascara indicador de los puntos de soporte usada para calcular la media y covarianza robusta. |
| raw_location_ | retorna la media sin reajuste. |
| raw_covariance_ | retona la matriz de covarianza sin reajuste. |
| raw_support_ | retorna una mascara indicador de los puntos de soporte usada para calcular la media y covarianza sin reajuste. | 
| dist_ | Distancia de mahalanobis del conjunto de entrenamiento. | 

<br>

|Funciones | Descripción |
|----------|-------------|
| fit(X) | Entrena el modelo con los parametros asignados.|
| fit_predict(X) | Entrena el modelo e identifica si una observación es outlier (-1) o no (1). |
| predict(X) | Identifica si una observación es outlier (-1) o no (1). |



In [ ]:
## Instancia del modelo
model = EllipticEnvelope(contamination = 0.1)

## Ajuste del modelo
model.fit(X_train)

## Etiquetado de las observación si es o no outliers
etiquetado = model.predict(X_train)

## Mostrar cantidad de outliers
print('Cantidad de outliers detectado: {}'.format((etiquetado == -1).sum()))

## Mostrar las etiquetas
etiquetado[:40]


#### Graficación de los puntos con etiquetas

In [ ]:
def confidence_ellipse(location, cov, ax, n_std=4, **kwargs):
    """
    Create a plot of the covariation confidence ellipse

    Returns
    -------
    float: the Pearson Correlation Coefficient

    Other parameters
    ----------------
    kwargs : `~matplotlib.patches.Patch` properties

    author : Carsten Schelp
    license: GNU General Public License v3.0 (https://github.com/CarstenSchelp/CarstenSchelp.github.io/blob/master/LICENSE)
    """
    
    #cov = np.cov(x, y)
    pearson = cov[0, 1]/sqrt(cov[0, 0] * cov[1,1])
    ell_radius_x = sqrt(1 + pearson)
    ell_radius_y = sqrt(1 - pearson)
    ellipse = Ellipse((0,0), width=ell_radius_x * 2, height=ell_radius_y * 2, **kwargs)

    # calculating the stdandarddeviation of x from  the squareroot of the variance
    # np.sqrt(cov[0, 0])
    scale_x = sqrt(cov[0, 0]) * n_std
    mean_x = location[0]
    
    # calculating the stdandarddeviation of y from  the squareroot of the variance
    # np.sqrt(cov[1, 1])
    scale_y = sqrt(cov[1, 1]) * n_std
    mean_y = location[1]
    
    transf = transforms.Affine2D() \
        .rotate_deg(45) \
        .scale(scale_x, scale_y) \
        .translate(mean_x, mean_y)
        
    ellipse.set_transform(transf + ax.transData)
    ax.add_patch(ellipse)
        
    return pearson        
    # render plot with "plt.show()".

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))

sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                hue=etiquetado,
                palette=sns.color_palette()[:2])

## Agregar forma de la matriz de covarianza robusta
_ = confidence_ellipse(model.location_, model.covariance_, ax, facecolor='none', edgecolor='red')

## Agregar forma de la matriz de covarianza real
#_ = confidence_ellipse(model.raw_location_, model.raw_covariance_, ax, facecolor='none', edgecolor='green', ls='--')

plt.legend(labels=["inliers", "outliers"], title="True class")
plt.title("Gaussian inliers with uniformly distributed outliers")
plt.tight_layout()
plt.show()


#### Si disponemos de información de los outliers provenientes de expertos. 

In [ ]:
plt.figure(figsize=(14, 7))
plt.subplot(1, 2, 1)
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                hue=etiquetado,
                palette=sns.color_palette()[:2])
plt.legend(labels=["inliers", "outliers"], title="True class")
plt.title("Robust Covariance")

plt.subplot(1, 2, 2)
sns.scatterplot(x=X_train[:, 0], 
                y=X_train[:,1], 
                hue=y_train,
                palette=sns.color_palette()[:2]
                )
plt.legend(labels=["inliers", "outliers"], title="True class")
plt.title("Reales")
plt.xlabel('x'), plt.ylabel('y')
plt.tight_layout()
plt.show()


In [ ]:
## Calculo de rendimiento del modelo si tenemos información de los outliers reales.
NroOutlier_predicha = (etiquetado == -1).sum()
NroOutlier_verdaderos = (y_train == -1).sum()
NroOutlier_acetados = ((y_train == -1) & (etiquetado == -1)).sum()
rateOutlier_acertados = ((y_train == -1) & (etiquetado == -1)).sum() / max(NroOutlier_predicha, NroOutlier_verdaderos)

print('Cantidad de outliers detectados: {}'.format(NroOutlier_predicha))
print('Cantidad de outliers verdaderos: {}'.format(NroOutlier_verdaderos))
print('Cantidad de outliers verdaderos-detectados: {}'.format(NroOutlier_acetados))
print('Porcentaje de outliers verdaderos-detectados: {:.2f}%'.format(100*rateOutlier_acertados))

## Búsqueda de la mejor configuración

In [ ]:
## Almacenador de resultados
output = {'contaminacion': [], 
          'rate_train_aciertos': [],
          'rate_val_aciertos': []}

## Lista de hiperparametros
lista_contaminacion = linspace(0.05, 0.5, 20)

for contaminacion in tqdm(lista_contaminacion):
    
    ## Instancia del modelo
    model = EllipticEnvelope(contamination = contaminacion, 
                             random_state=0)

    ## Ajuste del modelo
    model.fit(X_train)
    etiquetado_train = model.predict(X_train)
    etiquetado_val = model.predict(X_val)

    ## Porcentaje de puntos anómalos acertados con respecto a los verdaderos
    acertados_train = ((y_train == -1) & (etiquetado_train == -1)).sum() / max((etiquetado_train==-1).sum(),(y_train==-1).sum())
    acertados_val = ((y_val == -1) & (etiquetado_val == -1)).sum() / max((etiquetado_val==-1).sum(),(y_val==-1).sum())

    ## Almcenar los resultados
    output['contaminacion'].append(contaminacion)
    output['rate_train_aciertos'].append(acertados_train)
    output['rate_val_aciertos'].append(acertados_val)

output = DataFrame(output).sort_values(by=['rate_val_aciertos', 'contaminacion'], ascending=False)
output.head(10)


In [ ]:
## Instancia del modelo
model = EllipticEnvelope(contamination = 0.144737)

## Ajuste del modelo y Etiquetado de las observación si es o no outliers
model.fit(X_train_Val)
etiquetado = model.predict(X_train_Val)

## Mostrar resumen comparativo
NroOutlier_predicha = (etiquetado == -1).sum()
NroOutlier_verdaderos = (y_train_val == -1).sum()
NroOutlier_acetados = ((y_train_val == -1) & (etiquetado == -1)).sum()
rateOutlier_acertados = ((y_train_val == -1) & (etiquetado == -1)).sum() / max(NroOutlier_predicha, NroOutlier_verdaderos)

print('Cantidad de outliers detectados: {}'.format(NroOutlier_predicha))
print('Cantidad de outliers verdaderos: {}'.format(NroOutlier_verdaderos))
print('Cantidad de outliers verdaderos-detectados: {}'.format(NroOutlier_acetados))
print('Porcentaje de outliers verdaderos-detectados: {:.2f}%'.format(100*rateOutlier_acertados))

In [ ]:
## Detección de outliers en el conjunto de test
etiquetado_test = model.predict(X_test)
NroOutlier_predicha = (etiquetado_test == -1).sum()
NroOutlier_verdaderos = (y_test == -1).sum()
NroOutlier_acetados = ((y_test == -1) & (etiquetado_test == -1)).sum()
rateOutlier_acertados = ((y_test == -1) & (etiquetado_test == -1)).sum() / max(NroOutlier_predicha, NroOutlier_verdaderos)

## Mostrar resumen comparativo
print('Cantidad de outliers detectados: {}'.format(NroOutlier_predicha))
print('Cantidad de outliers verdaderos: {}'.format(NroOutlier_verdaderos))
print('Cantidad de outliers verdaderos-detectados: {}'.format(NroOutlier_acetados))
print('Porcentaje de outliers verdaderos-detectados: {:.2f}%'.format(100*rateOutlier_acertados))